# FAERS Adverse Event Demographic Data 
# EDA & Association Rule Mining

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

import networkx as nx

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

VIZ_COUNT = 0
def new_fig(title=""):
    """Helper to keep a running count of every visualization produced."""
    global VIZ_COUNT
    VIZ_COUNT += 1
    print(f"[Visualization #{VIZ_COUNT}] {title}")


## 2. Loading the Dataset

In [ ]:
df_q1 = pd.read_csv("demo2025q1.csv", low_memory=False)
df_q2 = pd.read_csv("demo2025q2.csv", low_memory=False)

print("Q1 shape:", df_q1.shape)
print("Q2 shape:", df_q2.shape)
df_q1.head()


In [ ]:
df_q1.info()


In [ ]:
df_q2.info()


## 3. Data Cleaning & Preparation

In [ ]:
# 3.1 Schema check

same_schema = list(df_q1.columns) == list(df_q2.columns)
print("Identical column schema across quarters:", same_schema)
print(df_q1.columns.tolist())

In [ ]:
# 3.2 Tag source quarter prior to merging

df_q1['quarter'] = '2025Q1'
df_q2['quarter'] = '2025Q2'

In [ ]:
# 3.3 Helper to convert an FAERS-style YYYYMMDD numeric date into a real datetime

def parse_faers_date(series):
    return pd.to_datetime(series, format='%Y%m%d', errors='coerce')

date_cols = ['event_dt', 'mfr_dt', 'init_fda_dt', 'fda_dt', 'rept_dt']

for d in [df_q1, df_q2]:
    for c in date_cols:
        d[c] = parse_faers_date(d[c])


In [ ]:
# 3.4 Standardize categorical missing values / blanks -> NaN, and trim whitespace

cat_cols = ['i_f_code', 'rept_cod', 'auth_num', 'mfr_num', 'mfr_sndr', 'lit_ref',
            'age_cod', 'age_grp', 'sex', 'e_sub', 'wt_cod', 'to_mfr', 'occp_cod',
            'reporter_country', 'occr_country']

for d in [df_q1, df_q2]:
    for c in cat_cols:
        d[c] = d[c].astype('string').str.strip()
        d[c] = d[c].replace({'': np.nan, 'nan': np.nan, 'UNK': np.nan})


In [ ]:
# 3.5 Sanity-clip biologically implausible age / weight values (data-entry errors)

for d in [df_q1, df_q2]:
    d.loc[(d['age'] < 0) | (d['age'] > 122), 'age'] = np.nan
    d.loc[(d['wt'] <= 0) | (d['wt'] > 400), 'wt'] = np.nan

## 4. Merge the Two Quarters into a Single Dataset

In [ ]:
merged_df = pd.concat([df_q1, df_q2], axis=0, ignore_index=True)
print("Merged shape before de-dup:", merged_df.shape)

before = len(merged_df)
merged_df = merged_df.drop_duplicates(subset=['primaryid', 'caseid', 'caseversion'])
after = len(merged_df)
print(f"Removed {before - after} exact duplicate case-versions across the two extracts.")
print("Final merged shape:", merged_df.shape)

merged_df.head()


In [ ]:
# Downcast categoricals for memory efficiency

for c in cat_cols + ['quarter']:
    merged_df[c] = merged_df[c].astype('category')

merged_df.info(memory_usage='deep')


## 5. Missing-Value 

In [ ]:
missing_pct = merged_df.isna().mean().sort_values(ascending=False) * 100
missing_pct = missing_pct[missing_pct > 0]
print(missing_pct.round(2))


In [ ]:
new_fig("Missing value percentage by column")
fig, ax = plt.subplots(figsize=(11, 8))
sns.barplot(x=missing_pct.values, y=missing_pct.index, palette='rocket', ax=ax)
ax.set_xlabel("% Missing")
ax.set_ylabel("Column")
ax.set_title("Missing Value Percentage by Column (Merged Dataset)")
for i, v in enumerate(missing_pct.values):
    ax.text(v + 0.5, i, f"{v:.1f}%", va='center', fontsize=9)
plt.tight_layout()
plt.show()


## 6. Exploratory Data Analysis (EDA)

### 6.1 Case Volume by Quarter

In [ ]:
new_fig("Number of cases per quarter")
fig, ax = plt.subplots()
counts = merged_df['quarter'].value_counts().sort_index()
sns.barplot(x=counts.index, y=counts.values, ax=ax, palette='mako')
ax.set_title("Number of Adverse Event Cases per Quarter")
ax.set_xlabel("Quarter")
ax.set_ylabel("Number of Cases")
for i, v in enumerate(counts.values):
    ax.text(i, v + 1000, f"{v:,}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


### 6.2 Sex Distribution

In [ ]:
new_fig("Sex distribution (pie)")
fig, ax = plt.subplots()
sex_counts = merged_df['sex'].value_counts(dropna=True)
colors = sns.color_palette('viridis', len(sex_counts))
ax.pie(sex_counts.values, labels=sex_counts.index, autopct='%1.1f%%',
       startangle=90, colors=colors, wedgeprops={'edgecolor':'white'})
ax.set_title("Patient Sex Distribution")
plt.tight_layout()
plt.show()


### 6.3 Age Distribution

In [ ]:
new_fig("Age distribution (histogram + KDE)")
fig, ax = plt.subplots()
sns.histplot(merged_df['age'].dropna(), bins=50, kde=True, color='teal', ax=ax)
ax.set_title("Patient Age Distribution")
ax.set_xlabel("Age (years)")
plt.tight_layout()
plt.show()


### 6.4 Age by Sex

In [ ]:
new_fig("Age distribution by sex (boxplot)")
fig, ax = plt.subplots()
plot_df = merged_df[merged_df['sex'].isin(['F', 'M'])]
sns.boxplot(data=plot_df, x='sex', y='age', palette='Set2', ax=ax)
ax.set_title("Age Distribution by Sex")
ax.set_xlabel("Sex")
ax.set_ylabel("Age (years)")
plt.tight_layout()
plt.show()


### 6.5 Age Group (`age_grp`) Distribution

In [ ]:
new_fig("Age group distribution")
fig, ax = plt.subplots()
age_grp_labels = {'N':'Neonate','I':'Infant','C':'Child','T':'Adolescent','A':'Adult','E':'Elderly'}
agc = merged_df['age_grp'].value_counts()
agc.index = [age_grp_labels.get(i, i) for i in agc.index]
sns.barplot(x=agc.values, y=agc.index, palette='crest', ax=ax)
ax.set_title("Age Group Distribution")
ax.set_xlabel("Number of Cases")
plt.tight_layout()
plt.show()


### 6.6 Patient Weight Distribution

In [ ]:
new_fig("Weight distribution (kg)")
fig, ax = plt.subplots()
wt_kg = merged_df.loc[merged_df['wt_cod'] == 'KG', 'wt'].dropna()
sns.histplot(wt_kg, bins=50, kde=True, color='indianred', ax=ax)
ax.set_title("Patient Weight Distribution (KG-reported cases)")
ax.set_xlabel("Weight (kg)")
plt.tight_layout()
plt.show()


### 6.7 Report Type (`rept_cod`) Distribution

In [ ]:
new_fig("Report type distribution")
fig, ax = plt.subplots()
rc = merged_df['rept_cod'].value_counts()
sns.barplot(x=rc.values, y=rc.index, palette='flare', ax=ax)
ax.set_title("Report Type Distribution")
ax.set_xlabel("Number of Cases")
plt.tight_layout()
plt.show()


### 6.8 Reporter Occupation (`occp_cod`) Distribution

In [ ]:
new_fig("Reporter occupation distribution")
occ_labels = {'MD':'Physician','CN':'Consumer','PH':'Pharmacist','HP':'Health Professional (other)','LW':'Lawyer'}
fig, ax = plt.subplots()
oc = merged_df['occp_cod'].value_counts()
oc.index = [occ_labels.get(i, i) for i in oc.index]
sns.barplot(x=oc.values, y=oc.index, palette='cubehelix', ax=ax)
ax.set_title("Reporter Occupation Distribution")
ax.set_xlabel("Number of Cases")
plt.tight_layout()
plt.show()


### 6.9 Top 15 Reporting Countries

In [ ]:
new_fig("Top 15 reporting countries")
fig, ax = plt.subplots(figsize=(10,7))
top_rc = merged_df['reporter_country'].value_counts().head(15)
sns.barplot(x=top_rc.values, y=top_rc.index, palette='viridis', ax=ax)
ax.set_title("Top 15 Reporting Countries")
ax.set_xlabel("Number of Cases")
plt.tight_layout()
plt.show()


### 6.10 Top 15 Countries Where the Event Occurred

In [ ]:
new_fig("Top 15 event-occurrence countries")
fig, ax = plt.subplots(figsize=(10,7))
top_oc = merged_df['occr_country'].value_counts().head(15)
sns.barplot(x=top_oc.values, y=top_oc.index, palette='magma', ax=ax)
ax.set_title("Top 15 Countries Where the Adverse Event Occurred")
ax.set_xlabel("Number of Cases")
plt.tight_layout()
plt.show()


### 6.11 Electronic Submission Flag (`e_sub`)

In [ ]:
new_fig("Electronic submission flag")
fig, ax = plt.subplots()
es = merged_df['e_sub'].value_counts()
ax.pie(es.values, labels=es.index, autopct='%1.1f%%', colors=sns.color_palette('pastel'),
       startangle=90, wedgeprops={'edgecolor':'white'})
ax.set_title("Electronically Submitted Reports (e_sub)")
plt.tight_layout()
plt.show()


### 6.12 Initial vs Follow-up Report (`i_f_code`)

In [ ]:
new_fig("Initial vs follow-up report flag")
fig, ax = plt.subplots()
ifc = merged_df['i_f_code'].value_counts()
sns.barplot(x=ifc.index, y=ifc.values, palette='Set2', ax=ax)
ax.set_title("Initial (I) vs Follow-up (F) Reports")
ax.set_ylabel("Number of Cases")
plt.tight_layout()
plt.show()


### 6.13 Case-Version Distribution (report amendment frequency)

In [ ]:
new_fig("Case version distribution")
fig, ax = plt.subplots()
cv = merged_df['caseversion'].clip(upper=10).value_counts().sort_index()
sns.barplot(x=cv.index, y=cv.values, palette='rocket', ax=ax)
ax.set_title("Case Version Distribution (capped at 10)")
ax.set_xlabel("Case Version #")
ax.set_ylabel("Number of Cases")
plt.tight_layout()
plt.show()


### 6.14 Monthly Trend of Adverse Event Onset (`event_dt`)

In [ ]:
new_fig("Monthly trend of adverse event onset dates")
event_monthly = (merged_df.dropna(subset=['event_dt'])
                  .set_index('event_dt')
                  .resample('ME').size())
event_monthly = event_monthly[event_monthly.index >= '2015-01-01']
fig, ax = plt.subplots(figsize=(13,5))
event_monthly.plot(ax=ax, color='darkslateblue')
ax.set_title("Monthly Count of Reported Adverse-Event Onset Dates (2015+)")
ax.set_xlabel("Event Date")
ax.set_ylabel("Number of Cases")
plt.tight_layout()
plt.show()


### 6.15 Monthly Trend of FDA Receipt Date (`fda_dt`)

In [ ]:
new_fig("Monthly trend of FDA receipt dates")
fda_monthly = merged_df.set_index('fda_dt').resample('ME').size()
fig, ax = plt.subplots(figsize=(13,5))
fda_monthly.plot(ax=ax, color='darkorange')
ax.set_title("Monthly Count of Reports Received by FDA")
ax.set_xlabel("FDA Receipt Date")
ax.set_ylabel("Number of Cases")
plt.tight_layout()
plt.show()


### 6.16 Correlation Heatmap (Numeric Fields)

In [ ]:
new_fig("Correlation heatmap of numeric fields")
numeric_cols = ['age', 'wt', 'caseversion']
corr = merged_df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title("Correlation Heatmap — Numeric Fields")
plt.tight_layout()
plt.show()


### 6.17 Age vs Weight Relationship

In [ ]:
new_fig("Age vs weight scatter")
sample_scatter = merged_df.dropna(subset=['age','wt']).sample(min(5000, merged_df.dropna(subset=['age','wt']).shape[0]), random_state=42)
fig, ax = plt.subplots()
sns.scatterplot(data=sample_scatter, x='age', y='wt', hue='sex', alpha=0.4, ax=ax, palette='Set1')
ax.set_title("Age vs Weight (sampled 5,000 cases)")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Weight (kg)")
plt.tight_layout()
plt.show()


### 6.18 Sex vs Report Type Cross-tab

In [ ]:
new_fig("Sex vs report type heatmap")
ct = pd.crosstab(merged_df['sex'], merged_df['rept_cod'])
fig, ax = plt.subplots(figsize=(8,5))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu', ax=ax)
ax.set_title("Sex vs Report Type — Case Counts")
plt.tight_layout()
plt.show()


### 6.19 Top 10 Reporting Manufacturers/Senders (`mfr_sndr`)

In [ ]:
new_fig("Top 10 reporting manufacturers")
fig, ax = plt.subplots(figsize=(10,6))
top_mfr = merged_df['mfr_sndr'].value_counts().head(10)
sns.barplot(x=top_mfr.values, y=top_mfr.index, palette='cividis', ax=ax)
ax.set_title("Top 10 Reporting Manufacturers / Senders")
ax.set_xlabel("Number of Cases")
plt.tight_layout()
plt.show()


### 6.20 Occupation vs Sex — Stacked Composition

In [ ]:
new_fig("Occupation vs sex stacked bar")
ct2 = pd.crosstab(merged_df['occp_cod'], merged_df['sex'], normalize='index') * 100
fig, ax = plt.subplots(figsize=(9,6))
ct2.plot(kind='bar', stacked=True, ax=ax, colormap='Spectral')
ax.set_title("Sex Composition (%) within Each Reporter Occupation")
ax.set_ylabel("% of Cases")
ax.set_xlabel("Reporter Occupation Code")
plt.legend(title='Sex', bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout()
plt.show()


### 6.21 Case Distribution: Quarter x Report Type

In [ ]:
new_fig("Quarter vs report type distribution")
ct3 = pd.crosstab(merged_df['quarter'], merged_df['rept_cod'])
fig, ax = plt.subplots(figsize=(9,5))
ct3.plot(kind='bar', ax=ax, colormap='tab10')
ax.set_title("Report Type Volume by Quarter")
ax.set_ylabel("Number of Cases")
plt.xticks(rotation=0)
plt.legend(title='Report Type', bbox_to_anchor=(1.02,1), loc='upper left')
plt.tight_layout()
plt.show()


### 6.22 EDA Summary Statistics
A consolidated descriptive-statistics table for all numeric and key categorical fields.


In [ ]:
display(merged_df[['age','wt','caseversion']].describe().T)
print()
display(merged_df[cat_cols + ['quarter']].describe().T)


## 7. Association Rule Mining

**Goal:** discover which combinations of case attributes (sex, age group, occupation, report type, submission type, country, etc.) tend to co-occur, using the **Apriori** algorithm.

Since FAERS demographic data is not naturally "transactional" (no drug/reaction basket here — that lives in other FAERS tables), we build a transactional representation by treating **each categorical value a case has** as an "item" in that case's basket — e.g. a single case might contain the items `{sex=F, age_grp=A, occp_cod=MD, rept_cod=EXP, e_sub=Y, country=US}`. Frequent itemsets and rules then reveal which demographic/administrative attributes are statistically associated with one another.

### 7.1 Build the Transaction Basket


In [ ]:
# Columns used to build the "basket" of items per case
basket_cols = ['sex', 'age_grp', 'occp_cod', 'rept_cod', 'e_sub', 'wt_cod', 'i_f_code']

# Bucket reporter_country into Top-9 + OTHER to keep item-space tractable
top_countries = merged_df['reporter_country'].value_counts().head(9).index
country_astype_str = merged_df['reporter_country'].astype('object')
country_series = country_astype_str.where(
    merged_df['reporter_country'].isin(top_countries), other='OTHER'
)
country_series = country_series.where(merged_df['reporter_country'].notna(), other=np.nan)

basket_source = merged_df[basket_cols].copy()
basket_source['reporter_country'] = country_series

# Prefix each value with its column name so items are unambiguous, e.g. "sex=F"
for c in basket_source.columns:
    basket_source[c] = c + '=' + basket_source[c].astype(str)
    basket_source.loc[merged_df[c].isna(), c] = np.nan  # keep true missing as NaN (excluded from basket)

# Drop cases missing more than half of the chosen attributes (too sparse to be informative)
basket_source = basket_source[basket_source.isna().sum(axis=1) <= 3]

print(f"Cases available for basket analysis: {len(basket_source):,}")


In [ ]:
# For computational tractability, sample a representative subset of transactions
N_SAMPLE = 60000
sample_baskets = basket_source.sample(n=min(N_SAMPLE, len(basket_source)), random_state=42)

transactions = sample_baskets.apply(lambda row: [v for v in row if isinstance(v, str)], axis=1).tolist()
print("Sample transaction:", transactions[0])
print("Number of transactions:", len(transactions))


In [ ]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
onehot_df = pd.DataFrame(te_array, columns=te.columns_)
print("One-hot item matrix shape:", onehot_df.shape)
onehot_df.head()


### 7.2 Frequent Itemsets (Apriori)

In [ ]:
frequent_itemsets = apriori(onehot_df, min_support=0.03, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False)
print(f"Number of frequent itemsets found: {len(frequent_itemsets)}")
frequent_itemsets.head(15)


### 7.3 Generate Association Rules & All Scores

We compute the full standard scorecard for every rule: **support, confidence, lift, leverage, conviction,** and **Zhang's metric**.

In [ ]:
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.1)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

# Readable item labels
rules['antecedents_str'] = rules['antecedents'].apply(lambda s: ', '.join(sorted(s)))
rules['consequents_str'] = rules['consequents'].apply(lambda s: ', '.join(sorted(s)))

print(f"Number of association rules generated: {len(rules)}")
score_cols = ['antecedents_str','consequents_str','support','confidence','lift','leverage','conviction','zhangs_metric']
rules[score_cols].head(20)


### 7.4 Top Rules — Interpretation Table
Sorted by **lift** (strength of association beyond random chance). Lift > 1 indicates a positive association.


In [ ]:
top_rules = rules[score_cols].head(20).copy()
for c in ['support','confidence','lift','leverage','conviction','zhangs_metric']:
    top_rules[c] = top_rules[c].round(3)
top_rules.columns = ['If (Antecedent)','Then (Consequent)','Support','Confidence','Lift','Leverage','Conviction',"Zhang's Metric"]
top_rules


## 8. Association-Rule Visualizations

### 8.1 Support vs Confidence (bubble size/color = Lift)

In [ ]:
new_fig("Support vs Confidence scatter (association rules)")
fig, ax = plt.subplots(figsize=(9,7))
sc = ax.scatter(rules['support'], rules['confidence'], c=rules['lift'], s=rules['lift']*40,
                 cmap='plasma', alpha=0.75, edgecolor='k')
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Lift')
ax.set_xlabel('Support')
ax.set_ylabel('Confidence')
ax.set_title('Association Rules: Support vs Confidence (bubble = Lift)')
plt.tight_layout()
plt.show()


### 8.2 Top 15 Rules by Lift

In [ ]:
new_fig("Top 15 rules by lift (bar chart)")
top15 = rules.head(15).copy()
top15['rule_label'] = top15['antecedents_str'] + '  ->  ' + top15['consequents_str']
fig, ax = plt.subplots(figsize=(11,8))
sns.barplot(x='lift', y='rule_label', data=top15, palette='viridis', ax=ax)
ax.set_title('Top 15 Association Rules Ranked by Lift')
ax.set_xlabel('Lift')
ax.set_ylabel('')
plt.tight_layout()
plt.show()


### 8.3 Rule Metric Correlation Heatmap

In [ ]:
new_fig("Correlation heatmap of rule scores")
metric_corr = rules[['support','confidence','lift','leverage','conviction','zhangs_metric']].corr()
fig, ax = plt.subplots(figsize=(7,6))
sns.heatmap(metric_corr, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Correlation Between Association-Rule Metrics')
plt.tight_layout()
plt.show()


### 8.4 Rule Network Graph (Top 12 Rules by Lift)

In [ ]:
new_fig("Network graph of top association rules")
G = nx.DiGraph()
top_net = rules.head(12)
for _, r in top_net.iterrows():
    for a in r['antecedents']:
        for c in r['consequents']:
            G.add_edge(a, c, weight=r['lift'])

fig, ax = plt.subplots(figsize=(11,9))
pos = nx.spring_layout(G, k=0.9, seed=42)
weights = [G[u][v]['weight'] for u,v in G.edges()]
nx.draw_networkx_nodes(G, pos, node_color='lightsteelblue', node_size=1800, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
edges = nx.draw_networkx_edges(G, pos, arrowstyle='-|>', arrowsize=15,
                                edge_color=weights, edge_cmap=plt.cm.autumn, width=2, ax=ax)
ax.set_title('Association Rule Network (Top 12 Rules by Lift)')
ax.axis('off')
plt.tight_layout()
plt.show()


### 8.5 Distribution of Lift Across All Discovered Rules

In [ ]:
new_fig("Distribution of lift values across all rules")
fig, ax = plt.subplots()
sns.histplot(rules['lift'], bins=30, kde=True, color='mediumseagreen', ax=ax)
ax.axvline(1, color='red', linestyle='--', label='Lift = 1 (independence)')
ax.set_title('Distribution of Lift Values Across All Association Rules')
ax.set_xlabel('Lift')
ax.legend()
plt.tight_layout()
plt.show()


**Total visualizations produced in this notebook: run the cell below to confirm the count (target: 20+).**

In [ ]:
print(f"Total visualizations generated: {VIZ_COUNT}")


## 9. Key Findings - All Scores at a Glance

| Metric | What it measures | How to read it |
|---|---|---|
| **Support** | How frequently the itemset appears in all cases | Higher = more common pattern |
| **Confidence** | P(consequent \| antecedent) | Higher = stronger predictive reliability |
| **Lift** | How much more likely the consequent is, given the antecedent, vs. chance | >1 = positive association, 1 = independent, <1 = negative |
| **Leverage** | Difference between observed and expected co-occurrence frequency | 0 = independence; further from 0 = stronger effect |
| **Conviction** | Ratio of expected frequency of antecedent without consequent to observed | Higher = antecedent strongly implies consequent |
| **Zhang's metric** | Normalized association strength, range [-1, 1] | Closer to 1 = strong positive association |


In [ ]:
summary_stats = rules[['support','confidence','lift','leverage','conviction','zhangs_metric']].describe().round(3)
summary_stats


## 10. Export the Final Merged Dataset

The cleaned & unified dataset (Q1 + Q2, de-duplicated, type-cast, dates parsed) is saved as a single compressed CSV for downstream use.


In [ ]:
output_path = "faers_2025_demo_merged.csv.gz"
merged_df.to_csv(output_path, index=False, compression='gzip')
print(f"Saved merged dataset -> {output_path}")
print(f"Final merged dataset shape: {merged_df.shape}")


## 11. Conclusion

- Two quarterly FAERS demographic extracts (**Q1 & Q2 2025**, ~400K and ~393K raw rows) were cleaned, standardized, and vertically merged into a single unified dataset of adverse-event case demographics.
- EDA revealed the typical FAERS reporting profile: a **majority-female**, **adult-to-elderly** patient population, reports dominated by the **US**, submitted mostly **electronically**, and largely filed as **Expedited (EXP)** or **Periodic (PER)** reports.
- Association Rule Mining on the categorical case attributes surfaced clear, high-lift relationships (e.g., between reporter occupation, report type, and submission method), all quantified with the full standard rule scorecard (support, confidence, lift, leverage, conviction, Zhang's metric).
- The merged, cleaned dataset has been exported and is ready for further modeling (e.g., signal detection, disproportionality analysis such as PRR/ROR against the companion FAERS drug/reaction tables).
